### Remove Trained Model

In [ ]:
from tensorflow.keras import backend as K
import gc

# Clear any previous model from memory
K.clear_session()

# Optionally, force garbage collection to free memory
gc.collect()

# Proceed with defining and training the new model


### PR Trained Model Using LSTM.

In [ ]:
import os
import tensorflow as tf
import numpy as np
import random
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, r2_score
import plotly.graph_objects as go

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)

# Force TensorFlow to run deterministically
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = ''

# Load and preprocess the data
data = pd.read_csv("../../../../data/vsf_prediction_model/demo/pr_lstm/pr_index.csv")
data['Date'] = pd.to_datetime(data['Date'], dayfirst=True)
data.set_index('Date', inplace=True)

# Scale the PR values between 0 and 1
scaler = MinMaxScaler()
data['PR_Scaled'] = scaler.fit_transform(data[['PR']])

# Create sequences for LSTM
def create_sequences(data, seq_length):
    sequences = []
    labels = []
    for i in range(len(data) - seq_length):
        seq = data[i:i + seq_length]
        label = data[i + seq_length]
        sequences.append(seq)
        labels.append(label)
    return np.array(sequences), np.array(labels)

seq_length = 30
data_array = data['PR_Scaled'].values
X, y = create_sequences(data_array, seq_length)

# Train-test split
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# Reshape for LSTM
X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

# Build the LSTM model with residual connection
model = Sequential([
    Bidirectional(LSTM(128, activation='relu', return_sequences=True, input_shape=(seq_length, 1), recurrent_dropout=0.2)),
    Dropout(0.2),
    Bidirectional(LSTM(64, activation='relu', return_sequences=False, recurrent_dropout=0.2)),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(1)  # Residual output
])

optimizer = Adam(learning_rate=0.001, clipnorm=1.0)
model.compile(optimizer=optimizer, loss='mse')

# Callbacks
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', patience=5, factor=0.5, verbose=1)
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)

# Train the model
history = model.fit(
    X_train, y_train,
    epochs=200,
    batch_size=16,
    validation_data=(X_test, y_test),
    callbacks=[lr_scheduler, early_stopping],
    verbose=1
)

# Save the trained model for consistent predictions
model.save("trained_lstm_model.h5")

# Evaluate the model
y_pred = model.predict(X_test)
y_pred_rescaled = scaler.inverse_transform(y_pred)
y_test_rescaled = scaler.inverse_transform(y_test.reshape(-1, 1))

mae = mean_absolute_error(y_test_rescaled, y_pred_rescaled)
r2 = r2_score(y_test_rescaled, y_pred_rescaled)

print(f"Mean Absolute Error (MAE): {mae}")
print(f"R^2 Score: {r2}")

# Predict future values
last_sequence = data_array[-seq_length:]
future_predictions = []

for _ in range(365):
    next_prediction = model.predict(last_sequence.reshape((1, seq_length, 1)))
    future_predictions.append(next_prediction[0][0])
    last_sequence = np.append(last_sequence[1:], next_prediction)

# Rescale future predictions
future_predictions_rescaled = scaler.inverse_transform(np.array(future_predictions).reshape(-1, 1))

# Create future predictions DataFrame
future_dates = pd.date_range(start=data.index[-1] + pd.Timedelta(days=1), periods=365, freq='D')
future_df = pd.DataFrame({'Date': future_dates, 'Predicted_PR': future_predictions_rescaled.flatten()})
future_df.set_index('Date', inplace=True)

# Save and plot predictions
future_df.to_csv("../../../../data/vsf_prediction_model/demo/pr_lstm/future_pr_predictions_improved.csv")

fig = go.Figure()
fig.add_trace(go.Scatter(x=future_df.index, y=future_df['Predicted_PR'], mode='lines', name='Predicted PR'))
fig.update_layout(
    title="Predicted PR for the Next Year (Improved Model)",
    xaxis_title="Date",
    yaxis_title="Predicted PR",
    legend=dict(x=0, y=1)
)
fig.show()


In [22]:
print(f"Mean Absolute Error (MAE): {mae}")
print(f"R^2 Score: {r2}")

Mean Absolute Error (MAE): 0.0005413373930835008
R^2 Score: 0.9129304680612346
